In [5]:
import pandas as pd
import numpy as np
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV

# --- Step 1: Load data ---
df = pd.read_csv("diabetes_data.csv")

df.rename(columns={"Gender": "Sex"}, inplace=True)
# Encode Sex column
df["Sex"] = df["Sex"].map({"Female": 0, "Male": 1})

df.columns = df.columns.str.replace(' ', '_')

# Encode binary columns (now with underscores in names)
binary_cols = ['Polyuria', 'Polydipsia', 'sudden_weight_loss',
               'weakness', 'Polyphagia', 'Genital_thrush', 'visual_blurring',
               'Itching', 'Irritability', 'delayed_healing', 'partial_paresis',
               'muscle_stiffness', 'Alopecia', 'Obesity']

for col in binary_cols:
    df[col] = df[col].map({'No': 0, 'Yes': 1})

# Encode target variable
df['class'] = df['class'].map({'Negative': 0, 'Positive': 1})

# --- Step 2: Balance dataset ---
df_no = df[df['class'] == 0]
df_yes = df[df['class'] == 1]

df_no_downsampled = resample(df_no,
                              replace=True,
                              n_samples=len(df_yes),
                              random_state=42)

df_balanced = pd.concat([df_no_downsampled, df_yes])

X = df_balanced.drop(columns=['class'])
y = df_balanced['class']

# --- Step 3: Train logistic regression ---
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
model.fit(X_train, y_train)

# --- Step 4: Calibrate probabilities ---
calibrated_model = CalibratedClassifierCV(model, cv="prefit")
calibrated_model.fit(X_val, y_val)

# --- Step 5: Adjust intercept to target baseline risk ---
# Get baseline probability for "all No" inputs
X_zero = np.zeros((1, X.shape[1]))
baseline_prob = calibrated_model.predict_proba(X_zero)[0][1]

target_prob = 0.08  # target baseline risk = 8%
delta = np.log(target_prob / (1 - target_prob)) - np.log(baseline_prob / (1 - baseline_prob))

model.intercept_ += delta

# Re-calibrate after intercept change
calibrated_model = CalibratedClassifierCV(model, cv="prefit")
calibrated_model.fit(X_val, y_val)

# --- Step 6: Test ---
print("Baseline risk (all No inputs):", calibrated_model.predict_proba(X_zero)[0][1])

# --- Step 7: Save model ---
import joblib
joblib.dump(calibrated_model, "diabetes_model_calibrated.pkl")


Baseline risk (all No inputs): 0.8819445544634154


C:\Users\Sarvesh\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\calibration.py:330: FutureWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(
C:\Users\Sarvesh\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
C:\Users\Sarvesh\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\calibration.py:330: FutureWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(
C:\Users\Sa

['diabetes_model_calibrated.pkl']